In [ ]:
from IPython.display import display, HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

### Our task is to develop a regression model that will predict the number of  crew members required for future ships from the given features.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

In [ ]:
spark = SparkSession.builder.appName('crew').getOrCreate()

### Read the data Crew.csv into spark dataframe
- inferSchema=True and header=True.
- Print the schema and show the first few rows.
- Use df.describe() to see the statistical properties of the data.

In [ ]:
df = spark.read.csv('Crew.csv', inferSchema=True, header=True)
df.printSchema()

root
 |-- Ship_name: string (nullable = true)
 |-- Cruise_line: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Tonnage: double (nullable = true)
 |-- passengers: double (nullable = true)
 |-- length: double (nullable = true)
 |-- cabins: double (nullable = true)
 |-- passenger_density: double (nullable = true)
 |-- crew: double (nullable = true)



In [ ]:
df.show(20)

+-----------+-----------+---+------------------+----------+------+------+-----------------+----+
|  Ship_name|Cruise_line|Age|           Tonnage|passengers|length|cabins|passenger_density|crew|
+-----------+-----------+---+------------------+----------+------+------+-----------------+----+
|    Journey|    Azamara|  6|30.276999999999997|      6.94|  5.94|  3.55|            42.64|3.55|
|      Quest|    Azamara|  6|30.276999999999997|      6.94|  5.94|  3.55|            42.64|3.55|
|Celebration|   Carnival| 26|            47.262|     14.86|  7.22|  7.43|             31.8| 6.7|
|   Conquest|   Carnival| 11|             110.0|     29.74|  9.53| 14.88|            36.99|19.1|
|    Destiny|   Carnival| 17|           101.353|     26.42|  8.92| 13.21|            38.36|10.0|
|    Ecstasy|   Carnival| 22|            70.367|     20.52|  8.55|  10.2|            34.29| 9.2|
|    Elation|   Carnival| 15|            70.367|     20.52|  8.55|  10.2|            34.29| 9.2|
|    Fantasy|   Carnival| 23| 

In [ ]:
df.describe().show()

+-------+---------+-----------+------------------+------------------+-----------------+-----------------+------------------+-----------------+-----------------+
|summary|Ship_name|Cruise_line|               Age|           Tonnage|       passengers|           length|            cabins|passenger_density|             crew|
+-------+---------+-----------+------------------+------------------+-----------------+-----------------+------------------+-----------------+-----------------+
|  count|      158|        158|               158|               158|              158|              158|               158|              158|              158|
|   mean| Infinity|       NULL|15.689873417721518| 71.28467088607599|18.45740506329114|8.130632911392404| 8.830000000000005|39.90094936708861|7.794177215189873|
| stddev|     NULL|       NULL| 7.615691058751413|37.229540025907866|9.677094775143416|1.793473548054825|4.4714172221480615| 8.63921711391542|3.503486564627034|
|    min|Adventure|    Azamara|   

### StringIndexer and OneHotEncoder
- Create StringIndexer and OneHotEncoder to process the data.
- StringIndexer is for any string data type.
- OneHotEncoder will be applied to the StringIndexer columns.
- Convert all obtained columns from OneHotEncoder and the other numeric columns into a feature column (use VectorAssembler)

In [ ]:
numCols = [s for (s, d) in df.dtypes if d != 'string']
numCols

['Age',
 'Tonnage',
 'passengers',
 'length',
 'cabins',
 'passenger_density',
 'crew']

In [ ]:
catCols = [s for (s, d) in df.dtypes if d == 'string' and s != 'Ship_name']
catCols

['Cruise_line']

In [ ]:
catCols_Ind = [s + '_Index' for s in catCols]
catCols_Ind

['Cruise_line_Index']

In [ ]:
catCols_OHE = [s + '_OHE' for s in catCols]
catCols_OHE

['Cruise_line_OHE']

In [ ]:
stind = StringIndexer(inputCols=catCols, outputCols=catCols_Ind, handleInvalid='skip')
ohe = OneHotEncoder(inputCols=catCols_Ind, outputCols=catCols_OHE)

In [ ]:
# Exclude label column 'crew' from feature columns
featureCols = catCols_OHE + [c for c in numCols if c != 'crew']
vecAssemb = VectorAssembler(inputCols=featureCols, outputCol='features')

### Divide the data into Train/Test

In [ ]:
trainDF, testDF = df.randomSplit([0.8, 0.2], seed=42)
print(f'There are {trainDF.count()} rows in the training set, and {testDF.count()} in the test set')

There are 133 rows in the training set, and 25 in the test set


### Create a Linear Regression Model

In [ ]:
lr = LinearRegression(featuresCol='features', labelCol='crew', predictionCol='prediction')

### Create a Pipeline model

In [ ]:
pl = Pipeline(stages=[stind, ohe, vecAssemb, lr])

### Fit the Pipeline model to the trainig data

In [ ]:
pl_Model = pl.fit(trainDF)

### Make a prediction for the same training data and evaluate the model performance using RMSE and r2

In [ ]:
predDF_train = pl_Model.transform(trainDF)

In [ ]:
predDF_train.show(5)

+---------+---------------+---+-------+----------+------+------+-----------------+-----+-----------------+---------------+--------------------+------------------+
|Ship_name|    Cruise_line|Age|Tonnage|passengers|length|cabins|passenger_density| crew|Cruise_line_Index|Cruise_line_OHE|            features|        prediction|
+---------+---------------+---+-------+----------+------+------+-----------------+-----+-----------------+---------------+--------------------+------------------+
|Adventure|Royal_Caribbean| 12|  138.0|     31.14|  10.2| 15.57|            44.32|11.85|              1.0| (18,[1],[1.0])|(24,[1,18,19,20,2...|12.268630903985958|
|  Allegra|          Costa| 21|  28.43|      8.08|  6.16|   4.1|            35.19|  4.0|              5.0| (18,[5],[1.0])|(24,[5,18,19,20,2...|3.4292848385416317|
|  Arcadia|            P&O|  9|   85.0|     19.68|  9.35|  9.84|            43.19| 8.69|             10.0|(18,[10],[1.0])|(24,[10,18,19,20,...| 9.186775011281973|
|    Aries|           

In [ ]:
rmse_eval = RegressionEvaluator(predictionCol='prediction', labelCol='crew', metricName='rmse')

In [ ]:
print('Train RMSE:', rmse_eval.evaluate(predDF_train))

Train RMSE: 0.8510804329072224


In [ ]:
r2_eval = RegressionEvaluator(predictionCol='prediction', labelCol='crew', metricName='r2')

In [ ]:
print('Train R2:', r2_eval.evaluate(predDF_train))

Train R2: 0.9422853579674988


### Make a prediction for the test data and evaluate the model performance using RMSE and r2

In [ ]:
predDF_test = pl_Model.transform(testDF)
print('Test RMSE:', rmse_eval.evaluate(predDF_test))

Test RMSE: 0.53797455463591


In [ ]:
print('Test R2:', r2_eval.evaluate(predDF_test))

Test R2: 0.9707651073148453
